## Import Dependencies

In [1]:
#import PySimpleGUI as sg
import pandas as pd 
import csv
import os
import matplotlib.pyplot as plt

import numpy as np
import math
import warnings
import re
import pprint
import inspect
import timeit
import pathlib
import sys
import seaborn as sns
pd.options.mode.chained_assignment = None

In [2]:
#import talos
from keras.models import Sequential
from keras.layers import Dropout, Dense
from keras.layers import LSTM    
from keras.regularizers import l2

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sys


2024-12-11 11:04:55.176976: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-11 11:04:55.192269: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-12-11 11:04:55.312244: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-12-11 11:04:55.396328: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1733940295.468513   10138 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1733940295.49

## Functions

In [5]:
def getDataCSV(city,variable,dtype):
##  dtype :  2 = return anomalies   1 = return normal, non-detrended data
    yrmos_detrend = []
    Stats = {}
    data_hold = {}
    data = pd.DataFrame()
    files = {}
    files["DENVER"] = "data/City/denver.csv"
    files["NEW ORLEANS"] = "data/City/neworleans.csv"
    files["NEW YORK"] = "data/City/newyork.csv"
    files["LOS ANGELES"] = "data/City/losangeles.csv"
    files["SAN FRANCISCO"] = "data/City/sanfrancisco.csv"
    
    data = pd.read_csv(files[city])
    # iterate over returned data from MondoDB and extract requested variable
    data["Month"] = data["Date"]
    data["Month"] = data["Month"].apply(getMonth)
    data["Date"] = data["Date"].str.replace("-","")
    data["Date"] = data["Date"].astype(int)

    tmax_vals=0
    tmin_vals=0
    prcp_vals=0
    tmax_miss=0
    tmin_miss=0
    prcp_miss=0
    lastrow=0
    years = {}
    for index,row in data.iterrows():
        yr = getYear(row["Date"])
        years[yr] = 1
        if (yr < 2021 and row["TMAX"] > -98 and row["TMIN"] > -98 and row["PRCP"] > -98):
            lastrow=index
    data = data.loc[0:lastrow,:]
# check to be sure there are no missing values
    tmin_miss = data[(data["TMIN"] < -98)].count()["TMIN"]
    tmax_miss = data[(data["TMAX"] < -98)].count()["TMAX"]
    prcp_miss = data[(data["PRCP"] < -98)].count()["PRCP"]
#    print(tmin_miss,tmax_miss,prcp_miss)
    if (tmin_miss > 0):
        print(f"MISSING TMIN VALUES {tmin_miss}  ... stopping so you can deal with it")
        exit()
    if (tmax_miss > 0):
        print(f"MISSING TMAX VALUES {tmax_miss}  ... stopping so you can deal with it")
        exit()
    if (prcp_miss > 0):
        print(f"MISSING PRCP VALUES {prcp_miss}  ... stopping so you can deal with it")
        exit()
    
    yrmos = data["Date"].to_list()
 #   print(data.tail(10))
#     reshaped data for input into neural netowrk
    y = data[variable].values.reshape(-1,1).astype(float)

    start = min(yrmos)
    end = max(yrmos)

    data_checks = pd.DataFrame()
    data_checks["Obs"] = data[variable]
    data_checks["Date"] = yrmos
    data_checks["Month"] = data["Month"]
    data_checks.insert(0,"Var",variable)
    if (dtype == 2):
        yrs=[]
        
        for yr in sorted (years.keys()):
            yrs.append(yr)

        months = ["","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
        data_bymonth = pd.DataFrame()
        data_bymonth["Year"] = yrs
        ilen = len(data_bymonth)
        for mon in range(1,13):
            temp = data[data["Month"] ==  mon]["TMIN"].to_list()
            nn = ilen - len(temp)
            if (nn > 0):
                print (f"{months[mon]} is short")
                for nn in range(1,nn+1):
                    temp.append(None)
            data_bymonth[months[mon]] = temp 
       

        months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

       
        Means = {}
        Stds = {}
        Mons = []
        months_int = {"Jan":1,"Feb":2,"Mar":3,"Apr":4,"May":5,"Jun":6,"Jul":7,"Aug":8,"Sep":9,
                     "Oct":10,"Nov":11,"Dec":12}
        data_checks["Means"] = None
        for mon in months:
             Stats[mon]={}
             data_bymonth[mon]=data_bymonth[mon].astype(float)
             Stats[mon]['std'] = data_bymonth[mon].std()
             Stats[mon]['mean'] = data_bymonth[mon].mean()
             Stds[mon] = (Stats[mon]['std'])
             Means[mon]=(Stats[mon]['mean'])
             Mons.append(mon)
             data_checks["Means"][data_checks["Month"] == months_int[mon]] = Means[mon]
    
       
        
        y_new=[]
        zz = []
        data_bymonth.reset_index()
        data_bymonth.columns = data_bymonth.columns.astype(str)
     #   print(data_bymonth.columns)

        
        
      #  print(data_bymonth.index)
        
        for index,row in data_bymonth.iterrows():
    #        for mon in range(0,12):
             for mon in months:
                if data_bymonth.loc[index,mon] > -98:
                    anom = data_bymonth.loc[index,mon] - Means[mon]
                    yy=[]
                    yy.append(anom)
                    zz.append(anom)
                    y_new.append(yy)
                    hld = []
                    mmint = months_int[mon]
                    hld.append(mon)
                    hld.append(row["Year"])
                    hld.append(anom)
                    yrmos_detrend.append(hld)
                 
        y = y_new
        data_checks["Anom"] = [yy[0] for yy in y]
        
 
    return y,yrmos,data_checks,start,end,yrmos_detrend



def  traceMsg(*inpt):
    
    nl = len(inspect.stack())
    nn=0
    going=True
    stck = []
    path = ""
    while (going and nn <= nl):
       pgm = inspect.stack()[nn][3]
       if (pgm != "<module>"):
         stck.append(pgm)
         if (pgm != "traceMsg"):
            path+=f"{pgm} <= "
       else:
          going=False
          path+=f"{pgm}"
       nn+=1
#    print (f"{inpt}  from  {path} ")

def getClimateIndices(directory):
    temp = []  # temporary list/dict used throught the program 
    features = pd.DataFrame()  # this is a Pandas Dataframe that will hold the Climate Indices

    #  add year-month pairs to the features dataframe that will match the available data
    for year in range(1950,2021):
        for month in range(1,13):
            yrmo = year*100 + month
            temp.append(yrmo)

    features['yearmo'] = temp
    with os.scandir(directory) as entries:
        for entry in entries:
            if entry.is_file():
             #   print(entry.name)
    #  the Climate Indices are stored in flat files, so read them all in and store them in features
                file=entry.name
                file = file.rstrip("\n")
                spl = file.split(".")
                name = spl[0]
                temp = []
                tempdf = pd.DataFrame()
                with open(f"data/Climate-Indices/{file}","r") as fin:
                    line=fin.readline()
                    spl = line.split()
                    start = int(spl[0])
                    end = int(spl[1])
                    nl=0
                    process=True
                    for nn in range(start,end+1):
                       line = fin.readline()
                       spl = line.split()
                       year = int(spl[0])
                       nl+=1
                       if (nl == 1 and year > 1950):  # make sure data starts in 1950 or earlier
                          process=False
                       if (year >= 1950):
                          for val in spl[1:]:
    #                       if float(val) != -99.99:
                            temp.append(val)
                 #   print(len(temp))
    #                if len(temp) == 828:
                    if (process):
                        features[name] = temp
                        features[name] = features[name].astype('float')
        #    temp["yearmo"] = features["yearmo"][(features["yearmo"] >= start) & (features['yearmo'] <= end)].astype(float)
        features["month"] =  features["yearmo"]
    #    features["month"] = features["month"].astype(str)
        for i in range(0, len(features)): 
            yrmn = features.loc[i,"month"]
            yrmn = str(yrmn)
            features.loc[i,"month"] = yrmn[4:]
        features["month"] = features["month"].astype(int)
 #       print(features.tail(10))   
        return features
    
def selectIndices(indicesString,features,start,end,dtype):
#  dtype  0 = return observations   1 = return anomalies 
    temp = pd.DataFrame()
    indices =indicesString.split(",")
 #   print("FEATURES ",features.head())
    for val in indices:
      val = val.lower()
#      print (val)
      temp[val] = features[val][(features["yearmo"] >= start) & (features['yearmo'] <= end)].astype(float)
     
    #  print(temp[val])
    if (dtype == 1):
      temp["month"] = features["month"][(features["yearmo"] >= start) & (features['yearmo'] <= end)].astype(float)
#      meanIndices = pd.DataFrame({"month":[mm for mm in range(1,13)]})
      means = {}
      for col in indices:
        means[col] = {}
        for mon in range(1,13):
          df = temp[col][temp["month"] == mon]
          mean = df.mean()
          means[col][mon] = mean
    
      
      for col in indices:
        colindex = temp.columns.get_loc(col)
        temp.insert(colindex,f"{col} Anom",np.nan)
        for i,row in temp.iterrows():
            val = row[col]
            mon = row["month"]
            row[f"{col} Anom"] = val-means[col][mon]
 #     pprint.pprint(means)
      for col in indices:
          temp  = temp.drop(col,axis=1)
      temp  = temp.drop("month",axis=1)      
 #     print(temp.head())
 
    nl = len(indices)
    X = temp.values.reshape(-1,nl).astype(float) 

    return (X)

def getMonth(a):
    a=str(a)
    b=a.split("-")
    return int(b[1])
def getYear(a):
    a=str(a)
#   b=a.split("-")
    return int(a[0:4])



 
    
def leastDen(a,b):
   
    c = a%b
    d = b-c
   
#    e=d%a
    return d,c


def trainFB(XXX,yyy,nsplit,toScale,phase,yrmos,fb,nbatch,nperiod,yrmos_d,data_checks):
    ''' create a Train, Test and Experiment datasets out of the input data. If toScale=1, then
    the data will also be scaled.  If toScale=0, the data will not be scaled.  The Experiment 
    time series will be the last nsplit values in the XX and yy lists. The Experiment time series 
    will be used for creating prediction scores'''
    
    info = {}
    info["y_test"] = {}
    info["y_train"] = {}
    ninst = len(yyy)  # length of input time series
##  adjust orignal TS for phase shift
    yy = yyy[phase:]
    yrmos_d = yrmos_d[phase:]
    yrmosShift = yrmos[phase:]
    XX = XXX[0:ninst-phase]
    ntlen=ninst
    ninst = len(yy)  # length of input time series
#    print ("LENS ",len(yy),len(XX))
    
## split off end of TS for experimental testing, if nsplit > 0
    y=yy[0:ninst-nsplit]  #  this will be used to create the Train and Test time sries for the dependant variables
    X=XX[0:ninst-nsplit]  #  this will be used to create the Train and Test time series for the independant variables
    yrmos_d = yrmos_d[0:ninst-nsplit]
    if (nsplit > 0):
        X_exp = XX[ninst-nsplit:]  # create the X experimental dataset for predictions
        y_exp = yy[ninst-nsplit:]  # create the y experimental dataset for predictions
    else:
        X_exp = []
        y_exp = []
    yrmosShift = yrmosShift[0:ninst-nsplit]
    feature_info = ""
   
## compute lenth of training time series 
    nlen = int(len(y)*.8)  # use 80% of data for training
#     print("NLEN ",nlen)
#  New method for train and test datea keeps time series togther instead of using random selected points
    if (fb == 1): # Train data from front of TS, testing at back
        X_train = X[:nlen] 
        X_test = X[nlen:] 
        y_train = y[:nlen] 
        y_test = y[nlen:] 
        
        info["y_train"]["start"] = phase
        info["y_train"]["end"] = phase+nlen-1
        info["y_test"]["start"] = phase+nlen
        info["y_test"]["end"] = phase+ninst-nsplit-1
        yrmosTrain = yrmosShift[:nlen]
        yrmosTest = yrmosShift[nlen:]    
        data_checks["Train"] = None
        data_checks["Test"] = None
    #    print ("CHKL ",len(y_train))
        data_checks.loc[phase:phase+len(y_train)-1,"Train"] = y_train 
        data_checks.loc[phase+len(y_train):,"Test"] = y_test 
    elif (fb == 2): # Train data from end of TS, testing data from front
        nl = len(y) - nlen
        
        X_train = X[nl:] 
        X_test = X[:nl] 
        y_train = y[nl:] 
        y_test = y[:nl] 
        info["y_train"]["start"] = phase+nl
        info["y_train"]["end"] =  phase+len(y)-1
        info["y_test"]["start"] = phase
        info["y_test"]["end"] = phase+nl-1
        yrmosTrain = yrmosShift[nl:]
        yrmosTest = yrmosShift[:nl]

    X_scaler = StandardScaler().fit(X_train)
    y_scaler = StandardScaler().fit(y_train)
    if (nsplit > 0):
        X_exp_scaled = X_scaler.transform(X_exp)
    else:
        X_exp_scaled = []
        
    if (toScale == 0):  # do not scale data, so return the unsclaled Train,Test and Experiment data
#        return (X_train,X_test,X_exp,y_train,y_test,y_exp,feature_info,X_scaler,y_scaler)
        X_train_scaled = np.array(X_train)
        X_test_scaled = np.array(X_test)
        y_train_scaled = np.array(y_train)
        y_test_scaled = np.array(y_test)
        return (info,yrmosTrain,yrmosTest,X_train_scaled,X_test_scaled,
        X_exp_scaled,y_train_scaled,y_test_scaled,y_exp,feature_info,
        X_scaler,y_scaler,y_train,y_test,nlen,ntlen)

    
#  create the scaled Train,Test and Predictio time series for the dependent variable    
    X_train_scaled = X_scaler.transform(X_train)
    X_test_scaled = X_scaler.transform(X_test)
 

#  create the scaled Train,Test and Predictio time series for the independent variable  
    y_train_scaled = y_scaler.transform(y_train)
    y_test_scaled = y_scaler.transform(y_test)
 #   y_exp_scaled = y_scaler.transform(y_exp)

    return (info,yrmosTrain,yrmosTest,X_train_scaled,X_test_scaled,X_exp_scaled,y_train_scaled, \
            y_test_scaled,y_exp,feature_info,X_scaler,y_scaler,y_train,y_test,nlen,ntlen,yrmos_d)

def reshapeTS(Xts,yts,nbatch,nperiod,how,yrmos,ino,ttp):
    
##  ttp = time series that is being reshaped... 1 = train, 2 = test
##  how = how to re-arrange the time series... either as
##        1 = [timesteps,features]
##        2 = [features,timesteps]
# yrmosTrain = yrmosTrain[nperiod:]
#     yrmosTest = yrmosTest[nperiod:]
#     y_train = y_train[nperiod:]
#     y_test = y_test[nperiod:]
#     X_train = X_train[nperiod:]
#     X_test = X_test[nperiod:]
#     info["y_train"]["start"]+=nperiod
#     info["y_test"]["start"]+=nperiod    
    traceMsg("INO ",ino)
    Xtsnew = []
    nlen = len(Xts)
    nfeats = Xts.shape[1]
#    traceMsg("start reshape ",nlen,nfeats,ts.shape,ts[0:10])
    nit=0
    if (how == 1):
        for ii in range(nperiod,nlen):
            tmp = []
            for jj in range(nit,ii+1):
                tmp.append(Xts[jj])
            Xtsnew.append(tmp)
            nit+=1
    elif (how == 2):
        for ii in range(nperiod,nlen):
            tmp = []
            for jj in range(0,nfeats):
                hld = []
                for mm in range(ii-nperiod,ii+1):
                    hld.append(Xts[mm][jj])
                tmp.append(hld)
                
            Xtsnew.append(tmp)    
  
    Xtsnew = np.array(Xtsnew)
    
    yts = yts[nperiod:]
    yrmos = yrmos[nperiod:]
    if (ttp == 1):
        ino["y_train"]["start"]+=nperiod
    elif (ttp ==2):
        ino["y_test"]["start"]+=nperiod
#    tsnew = tsnew.reshape(len(tsnew),nperiod+1,nfeats)
#    print("shape ",nperiod,tsnew[0:3])
#    npr = nperiod

## if using a batch_size, then make sure length of ts is divisible by batch_size    
##  
    if (nbatch > 0):
        newl,dif = leastDen(len(Xtsnew),nbatch)
        ol = len(Xtsnew)
#        print ("OLEN ",len(Xtsnew),len(yts),len(yrmos))
        Xtsnew = Xtsnew[dif:]
        yts = yts[dif:]
        yrmos=yrmos[dif:]
        if (ttp ==1):
             ino["y_train"]["start"]+= dif
        elif (ttp == 2):
             ino["y_test"]["start"]+= dif
    #    print("LLEN ",ol,nbatch,len(Xtsnew),len(yts),len(yrmos))


    return Xtsnew,yts,yrmos,ino


# path = os.path.abspath('')
# indices = "nina1,nina4,oni,tni"                              
#city = "DENVER"
#variable = "TMIN"
# yy,yrmos,data_checks,start,end,yrmos_d = getDataCSV("DENVER","TMIN",1)
# features = getClimateIndices(f"{path}/data/Climate-Indices/")

# XX = selectIndices(indices,features,start,end,1)

## neuralLSTMTT

In [6]:
def neuralLSTMTT(city,variable,indices,neuro_how,first_neuron,layer,dtype,fb,nperiod,
                 nbatches,nsplit,nphase,interactive,nepochs,nneurons):
#  dtype   -  0 = use actual obs   1 = use anaomalies
#  nphase  - amount of the phase shift
#  nperiod - how many time steps to go back in time for each point
#  ttype   - 1 = use Training data for stats   2 = use Test data for stats
#    nbatches = 8
#    nepochs = 100
#    nneurons= 100
    print ("PPERIOD ",nperiod)
 #   traceMsg("INFO  ",info)
    path = os.path.abspath('')
## get climate data
    features = getClimateIndices(f"{path}/data/Climate-Indices/")
 #   print("FFF ",features)
    
## get city data
    (y,yrmos,data_checks,start,end,yrmos_d) = getDataCSV(city,variable,dtype)
    X = selectIndices(indices,features,start,end,dtype)

    yy = [nn for nn in y]
## split training data
    (info,yrmosTrain,yrmosTest,X_train_scaled,X_test_scaled,X_exp_scaled,y_train_scaled, \
    y_test_scaled,y_exp_scaled,feature_info,X_scaler,y_scaler,
    y_train,y_test,nlen,ntlenk,yrmos_d) = trainFB(X,yy,nsplit,1,nphase,yrmos,fb,
                                                  nbatches,nperiod,yrmos_d,data_checks)
   
#    print ("CHECKS ",data_checks)
    y_train_scaled = y_train_scaled.reshape((y_train_scaled.shape[0]))
    y_test_scaled = y_test_scaled.reshape((y_test_scaled.shape[0]))
    X_train_scaled,y_train_scaled,yrmosTrain,info = \
    reshapeTS(X_train_scaled,y_train_scaled,nbatches,nperiod,2, \
    yrmosTrain,info,1)
    X_test_scaled,y_test_scaled,yrmosTest,info  = \
    reshapeTS(X_test_scaled,y_test_scaled,nbatches,nperiod,2,yrmosTest, \
    info,2)
    if (nsplit > 0):
        X_exp_scaled = reshapeTS(X_exp_scaled,nperiod)
    #y_train_scaled = y_train_scaled.reshape((y_train_scaled.shape[0], nperiod+1, y_train_scaled.shape[1]))
    #X_train_scaled = X_train_scaled.reshape((X_train_scaled.shape[0], nperiod+1, X_train_scaled.shape[1]))
   

#    traceMsg("SHAPE ",X_train_scaled.shape)
    model = Sequential()
#    print ("X_train",X_train_scaled[0:10])
#    print("y_train_scaled",y_train_scaled[0:10])
    nbatches_o = nbatches
    if (neuro_how == 1):
        
        model = Sequential()
        model.add(LSTM(nneurons, return_sequences=True,input_shape=(X_train_scaled.shape[1], 
        X_train_scaled.shape[2])))
        model.add(LSTM(nneurons, return_sequences=True))
        model.add(LSTM(nneurons))
        model.add(Dense(1))
        # model.add(Dense(1))
        # model.add(Dense(1))
        # model.add(Dense(1))
        # model.add(Dense(1))
        model.compile(loss='mae', optimizer='adam')
        r_train=0
        r_test=0
        
        history = model.fit(X_train_scaled, y_train_scaled, epochs=50, batch_size=24, validation_data=(X_test_scaled, y_test_scaled), verbose=0, shuffle=False)
        # plot history
        plt.plot(history.history['loss'], label='train')
        plt.plot(history.history['val_loss'], label='test')
        plt.legend()
        plt.show()
        if (nsplit > 0):
            y_predict = model.predict(X_exp_scaled)
            y_predict = y_scaler.inverse_transform(y_predict)
        y_train_predict = model.predict(X_train_scaled)
        y_train_predict = y_scaler.inverse_transform(y_train_predict)
   
        y_test_predict = model.predict(X_test_scaled)
        y_test_predict = y_scaler.inverse_transform(y_test_predict)
        y_train_scaled = y_scaler.inverse_transform(y_train_scaled)
        y_test_scaled = y_scaler.inverse_transform(y_test_scaled)
        y_train_predict = [mm[0] for mm in y_train_predict]
        y_test_predict = [mm[0] for mm in y_test_predict]
        y_train = [mm[0] for mm in y_train]
        y_test = [mm[0] for mm in y_test]
 #       print ("Chk Lens ",len(y_train),len(y_train_predict))
        r2_train=0
        r2_test=0
        
    elif (neuro_how == 2):  # use a non-1 batch size
#        y_train_scaled = y_train_scaled.reshape(1,-1) 
#        y_test_scaled = y_test_scaled.reshape(1,-1)

        model.add(LSTM(nneurons, stateful=True,batch_size=nbatches,kernel_regularizer=l2(0.30),
        batch_input_shape=(nbatches,X_train_scaled.shape[1], X_train_scaled.shape[2])))
        model.add(Dense(1))
        model.compile(loss='mae', optimizer='adam')
        print("Stage 2")
       
        for i in range(nepochs):
             model.fit(X_train_scaled, y_train_scaled, epochs=1, 
             batch_size=nbatches,verbose=0, shuffle=False)
             model.reset_states()
 #       print("y_train_scaled ",len(y_train_scaled),type(y_train_scaled),y_train_scaled.shape)
       
        y_train_predict = model.predict(X_train_scaled,batch_size=nbatches)
       
        r2_train=0
        y_train_predict = y_scaler.inverse_transform(y_train_predict)
        y_train_scaled = y_scaler.inverse_transform(y_train_scaled.reshape(-1,1))
        y_test_scaled = y_scaler.inverse_transform(y_test_scaled.reshape(-1,1))
        y_train_predict = [mm[0] for mm in y_train_predict]
        y_train = [mm[0] for mm in y_train] 
        y_train_scaled = [mm[0] for mm in y_train_scaled]    
        y_test_scaled = [mm[0] for mm in y_test_scaled]    
        
        nbatchs_o = nbatches
        nbatches=1
        new_model = Sequential()
        new_model.add(LSTM(nneurons, batch_input_shape=(nbatches, 
        X_train_scaled.shape[1], X_train_scaled.shape[2]), stateful=True))
        new_model.add(Dense(1))
        # copy weights
        old_weights = model.get_weights()
    
      
        new_model.set_weights(old_weights)
        # compile model
        new_model.compile(loss='mean_squared_error', optimizer='adam')        

        y_test_predict = np.array([])
#        print("yy_test_scaled ",y_test_scaled.shape,type(y_test_scaled))
#        print("Stage 2")
        for i in range(len(X_test_scaled)):
            testy =  y_test_scaled[i]
            testX = X_test_scaled[i]
            testX = testX.reshape(1, X_test_scaled.shape[1], X_test_scaled.shape[2])
            yhat = new_model.predict(testX, batch_size=1,verbose=0)
            y_test_predict = np.append(y_test_predict,yhat)
        
#        print("Shape ",y_test_predict.shape)
#        print(len(y_test_predict))
#        y_test_predict = np.array(y_test_predict)
        y_test_predict = y_scaler.inverse_transform(y_test_predict.reshape(-1,1))
#        y_test_predict = [mm[0][0] for mm in y_test_predict]
        r2_test=0
        y_test = [mm[0] for mm in y_test]
        y_test_predict = [mm[0] for mm in y_test_predict]

 



    return model,y_train_scaled,y_train_predict,y_test_scaled,y_test_predict,yrmosTrain,yrmosTest
 

## GUI

In [7]:
# Create some elements
def getDates(row):
    row["Year"] = int(row["Date"][:4])
    row["Month"] = int(row["Date"][4:])
    return row


def runApp(nit,parms):
#    neuro_how = 2  #  1 use old way, no batch size set, 2 = set batch size

#    indices = "nina1,nina4,oni,tni"  
    indices="ammsst,amon,epo,nao,nina1,nina3,nina4,pna,qbo,tna,tni,tsa,whwp,wp"    
    city = "DENVER"
    variable = "TMAX"
#    dtype = 0  #  1 = Actual monthly observations    2 = Monthly anomalies
    ttype = 2  #  1 = use Training data for stats    2 = using Testing data for stats
#    nphase=2
    nlayers=0
    nnodes=0
    #nsplit=24
    nsplit=0
#    nperiod = 5
#    nbatches = 4
    first_neuron = 50
    layer = 1
    fb=1
 #   print("calling nuearl")

#### New Parms 
    neuro_how = parms["neuro_how"]
    nphase = parms["phase"]
    nperiod = parms["period"]
    nbatches = parms["nbatch"]
    neurons = parms["neurons"]
    epochs = parms["epochs"]
    dtype = parms["dtype"]
    
    ###-----------------
    model,y_train_scaled,y_train_predict,y_test_scaled,y_test_predict,yrmosTrain,yrmosTest = \
    neuralLSTMTT(city,variable,indices,neuro_how,first_neuron,layer,
                                   dtype,fb,nperiod,nbatches,nsplit,nphase,interactive,neurons,epochs)
 #   print("Predicts ",y_test_predict)
  
    return model,y_train_scaled,y_train_predict,y_test_scaled,y_test_predict,yrmosTrain,yrmosTest
#    window["output"].update(new_summry)

###########  -----------------------------  #######################    
header_list = ["City","Var","NIT","Dtp","Phs","Wndw","Epcs","# Nrns","Neur","Btch","Tr Wi","Te Wi","Tr Sp","Te Sp","Tr Su","Te Su","Tr Fa","Te Fa"]
 
#window.close()

#neuro_h = 1
#dtype=2

#phase = int(values["phs"])
#period = int(values["wndw"])
#neurons = int(values["nrns"])
#epochs = int(values["epchs"])

interactive=2
#nrun+=1

neurons = 10  #

nit=0


params = {}
params["neuro_how"] = 2  # 1 = old way, no batch   2 = use batching
params["period"] = 30
params["phase"] = 1
params["epochs"] = 50
params["neurons"] = 50
params["dtype"] = 0 # 0 = Obs, 1 = anomalies
params["nbatch"] = 16

#params[""]
#params[""]
#params[""]
#params[""]
#params[""]
nit=0
nits=10
test_dfs=[]
train_dfs=[]

for ph in [8,9]:    #  BATCH SIZE
  for pp in [24]:  # PERIOD
    for aa in [150]: # NEURONS
        for eps in [150]: 
            for nit in range(nits):
            
                print("Period ",ph,pp,aa,eps,nit)
                params["period"] = pp
                params["neurons"] = aa
          #      params["nbatch"] = bb
                params["phase"] = ph   
                params["epochs"] = eps
                model,y_train_scaled,y_train_predict,y_test_scaled,y_test_predict,yrmosTrain,yrmosTest = \
                 runApp(nit,params)
                if nit == 0:
                    train_df = pd.DataFrame({"Date":yrmosTrain,"Obs":y_train_scaled,"Pred":y_train_predict})
                    train_df["Date"] = train_df["Date"].astype(str)
                    train_df = train_df.apply(getDates,axis=1)
                    # train_df["Diffs"] = train_df["Pred"] - train_df["Obs"]
                    # train_df["abs Diffs"] = abs(train_df["Pred"] - train_df["Obs"])

                #    print("test info ",len(yrmosTest),len(y_test_scaled),len(y_test_predict))

                    test_df = pd.DataFrame({"Date":yrmosTest,"Obs":y_test_scaled})
                    test_df["Date"] = test_df["Date"].astype(str)
                    test_df = test_df.apply(getDates,axis=1)
                    # test_df["Diffs"] = test_df["Pred"] - test_df["Obs"]
                    # test_df["abs Diffs"] = abs(test_df["Pred"] - test_df["Obs"])


                    for dd in params.keys():
                        train_df[dd]=params[dd]
                        test_df[dd]=params[dd]


                test_df[f"Pred {nit}"] = y_test_predict
                train_df[f"Pred {nit}"] = y_train_predict
            test_dfs.append(test_df)
            train_dfs.append(train_df)
            
            ttst= pd.concat(test_dfs)
           
            ttst.to_csv("Output\test_results_epochs_150_neurons_150_L30_ph8_9.csv",index=False)
            ttrn= pd.concat(train_dfs)
            ttrn.to_csv("Output\train_results_epochs_150_neurons_150_L30_ph8_9.csv",index=False)
                # summary_train = pd.concat([summary_train,train_df])
                    # summary_test = pd.concat([summary_test,test_df])

           #     else:
           #         summary_train = train_df
            #        summary_test = test_df


        

#summry.to_csv("DEN-TMIN-testing.csv",mode="a",header=False)

print(len(y_train_scaled),len(y_train_predict),len(yrmosTrain))
print("Done ")


Period  8 24 150 150 0
PPERIOD  24


/tmp/ipykernel_10138/2223718222.py:208: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '01' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  features.loc[i,"month"] = yrmn[4:]
W0000 00:00:1733940492.429908   10138 gpu_device.cc:2344] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


ValueError: Unrecognized keyword arguments passed to LSTM: {'batch_size': 16, 'batch_input_shape': (16, 14, 25)}

In [9]:
test_dfs

[       Date   Obs  Year  Month  neuro_how  period  phase  epochs  neurons  \
 0    200901  47.6  2009      1          2      24      8     150       50   
 1    200902  52.9  2009      2          2      24      8     150       50   
 2    200903  56.6  2009      3          2      24      8     150       50   
 3    200904  57.6  2009      4          2      24      8     150       50   
 4    200905  65.0  2009      5          2      24      8     150       50   
 ..      ...   ...   ...    ...        ...     ...    ...     ...      ...   
 139  202008  92.5  2020      8          2      24      8     150       50   
 140  202009  79.8  2020      9          2      24      8     150       50   
 141  202010  65.9  2020     10          2      24      8     150       50   
 142  202011  58.8  2020     11          2      24      8     150       50   
 143  202012  47.3  2020     12          2      24      8     150       50   
 
      dtype  ...     Pred 0     Pred 1     Pred 2     Pred 3  

In [19]:
ttst= pd.concat(test_dfs)
file_path = r"\\wsl$\Ubuntu\home\joe\work\Fire\Output\example.csv"

ttst.to_csv(file_path,index=False)

In [ ]:
ttrn= pd.concat(test_dfs)
ttrn.to_csv("train_results_Per12-24_neurons_100-250.csv",index=False)

In [8]:
! dir \\wsl.localhost\Ubuntu

 Volume in drive \\wsl.localhost\Ubuntu has no label.

 Directory of \\wsl.localhost\Ubuntu

06/17/2023  12:14 PM    <DIR>          wslpbJOFm
12/10/2024  07:55 AM    <DIR>          ..
09/15/2023  11:56 AM    <DIR>          snap
12/10/2024  07:55 AM    <DIR>          .
12/10/2024  07:55 AM    <DIR>          dev
06/17/2023  12:14 PM    <DIR>          wslhIccgI
08/15/2024  03:34 PM    <DIR>          root
01/03/2023  02:40 PM    <DIR>          srv
01/03/2023  02:40 PM    <JUNCTION>     lib [...]
07/25/2024  01:15 PM    <DIR>          var
09/28/2023  01:56 PM    <DIR>          wslaEkhJC
09/28/2023  01:56 PM    <DIR>          wslGnKnFD
03/24/2023  06:33 AM    <DIR>          mnt
07/12/2023  08:24 PM    <DIR>          wsloGDpnn
09/29/2023  09:15 AM    <DIR>          wslCBJohe
05/13/2024  08:09 PM    <DIR>          wslkbgBIi
09/15/2023  11:57 AM    <DIR>          boot
05/13/2024  07:38 PM    <DIR>          wslopKLeO
12/10/2024  07:55 AM    <DIR>          proc
01/03/2023  02:40 PM    <JUNCTION> 

In [ ]:
ttst["period"].value_counts()

In [ ]:
train_df["neurons"].value_count()

In [ ]:
 cols = ['Pred 1', 'Pred 2',
       'Pred 3', 'Pred 4', 'Pred 5', 'Pred 6', 'Pred 7', 'Pred 8', 'Pred 9',
       'Pred 10', 'Pred 11', 'Pred 12', 'Pred 13', 'Pred 14', 'Pred 15',
       'Pred 16', 'Pred 17', 'Pred 18', 'Pred 19', 'Pred 20', 'Pred 21',
       'Pred 22', 'Pred 23', 'Pred 24', 'Pred 25', 'Pred 26', 'Pred 27',
       'Pred 28', 'Pred 29']
    
 cols = ['Pred 1', 'Pred 2',
       'Pred 3', 'Pred 4', 'Pred 5', 'Pred 6', 'Pred 7', 'Pred 8', 'Pred 9']

In [ ]:
cols = ['Pred 1', 'Pred 2',
       'Pred 3', 'Pred 4', 'Pred 5', 'Pred 6', 'Pred 7', 'Pred 8', 'Pred 9']

def ensMeans(row):
    sums=0
    for col in cols:
        sums+= row[col]
    mean=sums/len(cols)
    return mean

ttst["Ens Mean"] = ttst.apply(ensMeans,axis=1)
ttrn["Ens Mean"] = ttrn.apply(ensMeans,axis=1)
ttst["New Diff"] = abs(ttst["Obs"]-ttst["Ens Mean"])
ttrn["New Diff"] = abs(ttrn["Obs"]-ttrn["Ens Mean"])                          

In [ ]:
ttst.to_csv("test_results_Per12-24_neurons_100-250.csv",index=False)
ttrn.to_csv("train_results_Per12-24_neurons_100-250.csv",index=False)

In [ ]:
print(test_df["New Diff"].mean())
print(train_df["New Diff"].mean())

In [ ]:
ttst.groupby(["period","neurons","Month"])["New Diff"].mean()

In [ ]:
display(ttrn.groupby(["period","neurons","Month"])["New Diff"].mean())

In [ ]:
test_df.head()

In [ ]:
test_df.plot(x="Date",y="New Diff")

In [ ]:
train_df.plot(x="Date",y="New Diff")

In [ ]:
train_df
test_df
all_df = pd.concat([train_df,test_df])

In [ ]:
all_df

In [ ]:
all_df.groupby("Month").agg({"Obs":["mean","std"]})

In [ ]:
cols=[]
train_df["Pred Avg"] = 0
for nn in range(29):
    col = f"Pred {nn}"
    cols.append(col)
    
for col in cols:
    train_df["Pred Avg"]+=train_df[col]
    
train_df["Pred Avg"]= train_df["Pred Avg"]/len(cols)

train_df["Avg Diff"] = abs(train_df["Obs"] - train_df["Pred Avg"]) 

print(train_df["Avg Diff"].mean())

In [ ]:
cols=[]
test_df["Pred Avg"] = 0
for nn in range(29):
    col = f"Pred {nn}"
    cols.append(col)
    
for col in cols:
    test_df["Pred Avg"]+=test_df[col]
    
test_df["Pred Avg"]= test_df["Pred Avg"]/len(cols)

test_df["Avg Diff"] = abs(test_df["Obs"] - test_df["Pred Avg"]) 

print(test_df["Avg Diff"].mean())

In [ ]:
test_df.groupby("Month").agg({"Avg Diff":["mean","std"]})


In [ ]:
#summary_test.loc[(summary_test["Month"] == 1) & (summary_test["Date"] == "201601") ]
#summary_train.loc[(summary_train["Month"] == 1) & (summary_train["Date"] == "200001") ]
print(summary_train.head())
sum_std = summary_train.groupby(["Date","nbatch"],as_index=False).std()
#summary_train.groupby(["nbatch"]).std()

In [ ]:
#summary_train["Date"].value_counts()
#sum_std
sum_std.loc[sum_std["Date"] == "198101"]

#sum_std.groupby("nbatch").mean()

In [ ]:
train_grouped = summary_train.groupby(["Month","period","neurons"],as_index=False).mean()
#train_grouped.head()

test_grouped = summary_test.groupby(["Month","period","neurons"],as_index=False).mean()
#print(test_grouped.head())

combined = train_grouped[["Month","period","epochs","neuro_how","phase","nbatch","neurons","dtype"]]
combined["Train absDiffs"] = train_grouped["abs Diffs"]
combined["Test absDiffs"] = test_grouped["abs Diffs"]

print(combined.head())

In [ ]:
# for per in train_grouped["Month"].unique():
#     g = sns.FacetGrid(train_grouped[train_grouped["Month"] == per], col="neurons",hue="phase")
#     g.map(sns.lineplot, "period", "abs Diffs", alpha=.7)

g = sns.FacetGrid(combined, col="period",row="Month")
g.map(sns.lineplot, "neurons", "Train absDiffs", alpha=.7)
g.map(sns.lineplot, "neurons", "Test absDiffs", alpha=.7)


In [ ]:
test_grouped = summary.groupby(["Month","period","epochs"],as_index=False).mean()
test_grouped.head()
g = sns.FacetGrid(test_grouped, col="epochs",row="Month")
g.map(sns.lineplot, "period", "abs Diffs", alpha=.7)

In [ ]:
sns.pairplot(train_grouped)

In [ ]:
train_grouped.head()
for month in range(1,13):
    train_grouped[(train_grouped["Month"] == month)].plot("period","abs Diffs")

In [ ]:
nrun=0

## testing

In [ ]:
neuro_h=1    # 1 = old way, no batch   2 = use batching
phase=3
period=30
nbatch=4
interactive=2
#nrun+=1
phases = [1]
neurons = 50
epochs=100
nit=0
dtype = 2
summry = pd.DataFrame()
summry_all = pd.DataFrame()
temp = pd.DataFrame()
for phase in [6]:
    for epochs in [50]:
#    for epochs in [100]:    
     for neurons in [40]:
#     for neurons in [20,40,80]:
#        for nbatch in [1,2,3,4]:
#            nit+=1
            print (f"Phs {phase} eps {epochs}  Nrn {neurons}  Btc {nbatch}")
            summry, new_list,nit,y_train,y_train_predict,df_train,df_test = runApp(neurons,epochs,nbatch,neuro_h,phase,period,nit,summry,interactive,dtype)
            summry.to_csv("DEN-TMIN.csv",mode="a",header=False)

            if nit > 1:
                summry_all = summry_all.append(summry)
                print (nit,"MERGING MERGING MERGING")
            else:
                summry_all = summry
#                temp[f"Obs {nit}"] = y_train
#                temp[f"Prd {nit}"] = y_train_predict
      
#temp["diffs"] = temp["Obs 1"] - temp["Prd 1"]   
#temp["adiffs"] = abs(temp["Obs 1"] - temp["Prd 1"] )   
#print (temp["adiffs"].mean())
#print (temp.head())
print(df_test.head())
print(summry)
#summry.to_csv("DEN-TMIN-testing.csv",mode="a",header=False)
x = [xx for xx in range(len(summry))]

sg.Popup(f' finished', keep_on_top=True)

## Save Models

In [ ]:
neuro_h=1    # 1 = old way, no batch   2 = use batching
phase=3
period=30
nbatch=4
interactive=2
neurons = 50
epochs=100
nit=0
dtype = 2
summry = pd.DataFrame()
summry_all = pd.DataFrame()
temp = pd.DataFrame()


print (f"Phs {phase} eps {epochs}  Nrn {neurons}  Btc {nbatch}")
model,summry, new_list,nit,y_train,y_train_predict,df_train,df_test = runApp(neurons,epochs,nbatch,neuro_h,phase,period,nit,summry,interactive,dtype)
summry.to_csv("DEN-TMIN-Finals.csv",mode="a",header=True)
model.save(f"models/model-LSTM-Den-Tmin-nh{neuro_h}-b{nbatch}-p{phase}-e{epochs}-n{neurons}")


In [ ]:
neuro_h=2    # 1 = old way, no batch   2 = use batching
phase=3
period=30
nbatch=4
interactive=2
#nrun+=1
phases = [1]
neurons = 50
epochs=100
nit=0
dtype = 2
summry = pd.DataFrame()
summry_all = pd.DataFrame()
temp = pd.DataFrame()
for phase in [0]:
    for epochs in [50]:
      for neurons in [5]:
        for nbatch in [3,4,5]:
#            nit+=1
            print (f"Phs {phase} eps {epochs}  Nrn {neurons}  Btc {nbatch}")
            summry, new_list,nit,y_train,y_train_predict,df_train,df_test = runApp(neurons,epochs,nbatch,neuro_h,phase,period,nit,summry,interactive,dtype)
#            summry.to_csv("DEN-TMIN.csv",mode="a",header=False)

            if nit > 1:
                summry_all = summry_all.append(summry)
                print (nit,"MERGING MERGING MERGING")
            else:
                summry_all = summry
#                temp[f"Obs {nit}"] = y_train
#                temp[f"Prd {nit}"] = y_train_predict
      
#temp["diffs"] = temp["Obs 1"] - temp["Prd 1"]   
#temp["adiffs"] = abs(temp["Obs 1"] - temp["Prd 1"] )   
#print (temp["adiffs"].mean())
#print (temp.head())
print(df_test.head())
print(summry)
#summry.to_csv("DEN-TMIN-testing.csv",mode="a",header=False)
x = [xx for xx in range(len(summry))]

sg.Popup(f' finished', keep_on_top=True)

In [ ]:
City   Var  NIT  Dtp  Phase  Period  Eps  # Nrn Neuro_How Batches  ...  \
0  DENVER  TMIN    0  Det      3      30  100     50       NoB       -  ...   

  Tr Fa ST Te Fa ST Tr Wi OP  Te Wi OP  Tr Sp OP Te Sp OP  Tr Su OP  Te Su OP  \
0     0.48     0.93  1.97657  3.390689  1.247085  2.45759  0.956326  2.396294   

   Tr Fa OP  Te Fa OP  
0   1.28507  2.662016  

In [ ]:
tss = ["Tr Wi","Tr Sp","Tr Su","Tr Fa"]


yy = []
for ts in tss:
    yh= summry[ts].astype(float).tolist()
    yy.append(yh)

xx = [x for x in range(len(yy[0]))]
def plotTS(xx,yy,file,title,labls):   
    colors = ["blue","green","orange","brown","black","red"]
    markers = ["s","o","v","8","p","*"]
    fig, ax1 = plt.subplots()
    for nn in range(0,len(yy)):
        ax1.plot(xx,yy[nn],label=labls[nn],marker=markers[nn],color=colors[nn])
    ax1.legend(loc="upper left")
    ax1.set_ylabel('Score')  
    ax1.set_xlabel('Phase Shift')  
    ax1.set_title(title)
    plt.show()
    if (file):
        fig.savefig(f"Plots/{file}")
                    
plotTS(xx,yy,f"DEN-TMIN-NOB-{period}-{nrun}.png",f"{nrun} Nit Scores for TMIN Train by Phase Shift for No Batch LSTM, Window Sz {period}",tss)

In [ ]:
sg.Popup('Ok clicked', keep_on_top=True)